In [36]:
import torch, sys

print("Python executable:", sys.executable)
print("Torch version     :", torch.__version__)
print("CUDA toolkit      :", torch.version.cuda)
print("CUDA available?   :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU name          :", torch.cuda.get_device_name(0))

Python executable: /opt/conda/bin/python3.11
Torch version     : 2.7.0+cu128
CUDA toolkit      : 12.8
CUDA available?   : True
GPU name          : NVIDIA GeForce RTX 5090 Laptop GPU


# Analyse des données énergétiques

Ce notebook contient l'analyse des données énergétiques normalisées et transformées.

## 1. Configuration et imports

In [37]:
import os
import json
import pandas as pd
from pandas.api.types import CategoricalDtype
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats
from scipy.stats import boxcox, probplot, fisher_exact, chi2_contingency, shapiro, t, normaltest, jarque_bera, anderson

from scipy.special import inv_boxcox

from sklearn.model_selection import train_test_split, cross_val_predict, RepeatedKFold, cross_val_score, GridSearchCV, ParameterGrid, KFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.compose import ColumnTransformer
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge, LinearRegression
from sklearn.utils import resample
from xgboost import XGBRegressor
import lightgbm as lgb

import statsmodels.api as sm
from statsmodels.api import OLS, WLS, add_constant
from statsmodels.stats.stattools import durbin_watson
from statsmodels.stats.diagnostic import het_breuschpagan
from statsmodels.stats.outliers_influence import outlier_test, OLSInfluence

import openai
from IPython.display import display, Markdown

import matplotlib.pyplot as plt

from dotenv import load_dotenv

import pprint

# Configuration des chemins
DATA_DIR = os.path.join('..', 'data')
NORMALIZED_DIR = os.path.join(DATA_DIR, 'normalized_csv')
TRANSFORMED_DIR = os.path.join(DATA_DIR, 'transformed_data')

# Afficher les chemins
print("DATA_DIR:", os.path.abspath(DATA_DIR))
print("NORMALIZED_DIR:", os.path.abspath(NORMALIZED_DIR))
print("TRANSFORMED_DIR:", os.path.abspath(TRANSFORMED_DIR))

# Vérifier l'existence des répertoires
print("Vérification des répertoires:")
print("DATA_DIR existe:", os.path.exists(DATA_DIR))
print("NORMALIZED_DIR existe:", os.path.exists(NORMALIZED_DIR))
print("TRANSFORMED_DIR existe:", os.path.exists(TRANSFORMED_DIR))

# Configuration de l'affichage
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
plt.style.use('default')  # Style par défaut de matplotlib
sns.set_theme()  # Style par défaut de seaborn

DATA_DIR: /workspace/Conso_Energy/data
NORMALIZED_DIR: /workspace/Conso_Energy/data/normalized_csv
TRANSFORMED_DIR: /workspace/Conso_Energy/data/transformed_data
Vérification des répertoires:
DATA_DIR existe: True
NORMALIZED_DIR existe: True
TRANSFORMED_DIR existe: True


## 2. Chargement des données

In [38]:
# Charger les données avec les bons paramètres
df = pd.read_csv(NORMALIZED_DIR+'/normalized_ENB2012_data.csv', sep=';',decimal=',')


In [39]:
# Fonction pour convertir les colonnes selon leur type
def convert_columns(df, column_config):
    """
    Convertit les colonnes selon leur type et applique les préfixes appropriés.
    
    Args:
        df (pd.DataFrame): DataFrame à convertir
        column_config (dict): Configuration des colonnes
        
    Returns:
        pd.DataFrame: DataFrame converti
    """
    # Créer une copie du DataFrame
    df_converted = df.copy()
    
    # Dictionnaire pour mapper les noms originaux aux noms avec préfixes
    column_mapping = {}
    
    # Parcourir la configuration et convertir les colonnes
    for code, config in column_config.items():
        original_name = config['name']
        new_name = f"{config['prefix']}{original_name}"
        
        if original_name in df_converted.columns:
            # Convertir selon le type
            if config['type'] == 'numeric':
                df_converted[original_name] = pd.to_numeric(df_converted[original_name], errors='coerce')
            elif config['type'] == 'categorical':
                df_converted[original_name] = df_converted[original_name].astype('category')
            
            # Enregistrer le mapping
            column_mapping[original_name] = new_name
    
    # Renommer les colonnes avec les préfixes
    df_converted = df_converted.rename(columns=column_mapping)
    
    return df_converted

# Convertir les données normalisées
print("Conversion des données normalisées :")
df_converted = convert_columns(df, column_config)
normalized_data_converted[file_name] = df_converted
print("Colonnes converties :")
display(df_converted.head())
print("Types des colonnes après conversion :")
display(df_converted.dtypes)

Conversion des données normalisées :
Colonnes converties :


,f_relative_compactness,f_surface_area,f_wall_area,f_roof_area,f_overall_height,f_orientation,f_glazing_area,f_glazing_distribution,l_heating_load,l_cooling_load
0,0.98,514.5,294.0,110.25,7.0,2,0.0,0,15.55,21.33
1,0.98,514.5,294.0,110.25,7.0,3,0.0,0,15.55,21.33
2,0.98,514.5,294.0,110.25,7.0,4,0.0,0,15.55,21.33
3,0.98,514.5,294.0,110.25,7.0,5,0.0,0,15.55,21.33
4,0.90,563.5,318.5,122.50,7.0,2,0.0,0,20.84,28.28


Types des colonnes après conversion :


f_relative_compactness     float64
f_surface_area             float64
f_wall_area                float64
f_roof_area                float64
f_overall_height           float64
f_orientation             category
f_glazing_area             float64
f_glazing_distribution    category
l_heating_load             float64
l_cooling_load             float64
dtype: object

## 3. Sélection des features

### Démarche d'analyse
Nous allons appliquer l'algorithme de sélection des features en trois étapes :

1. **Sélection initiale des features numériques** :
   - Calcul de la somme des corrélations absolues avec tous les labels pour chaque feature
   - Tri des features par ordre décroissant de cette somme (les plus corrélées d'abord)
   - Sélection itérative en évitant la multicolinéarité :
     * On prend la première feature (la plus corrélée avec les labels)
     * Pour chaque feature restante, on la garde uniquement si sa corrélation avec toutes les features déjà sélectionnées est ≤ 0.85
     * Cela permet d'éviter la redondance d'information tout en gardant les features les plus pertinentes

2. **Sélection des features catégorielles** :
   - Calcul d'un score de "corrélation" pour chaque feature catégorielle :
     * Pour chaque label, on effectue un test statistique (Wilcoxon ou Kruskal-Wallis selon le nombre de classes)
     * Le score est la somme des -log(p-value) pour tous les labels
     * Plus le score est élevé, plus la feature est liée aux labels
   - Tri des features par score décroissant
   - Élimination des redondances :
     * Test de Fisher entre chaque paire de features catégorielles
     * Si dépendance détectée (p-value < 0.05), on garde celle avec le meilleur score

3. **Validation finale** :
   - Pour chaque paire (feature numérique, feature catégorielle) :
     * Test statistique pour vérifier leur indépendance
     * Si dépendance détectée (p-value < 0.05), on élimine la feature catégorielle
     * Cela permet d'éviter la redondance entre types de features différents


In [41]:
import pandas as pd
import numpy as np
from scipy import stats
from scipy.stats import chi2_contingency, fisher_exact

# Initialiser les ensembles pour stocker les features
features_to_keep = set()
features_to_eliminate = set()

print(f"\n{'='*80}")
print(f"Sélection des features pour {file_name}")
print(f"{'='*80}")

# Identifier les features numériques et catégorielles
numeric_features = [col for col in df_converted.columns 
                    if col.startswith('f_') and df_converted[col].dtype in ['int64', 'float64']]
categorical_features = [col for col in df_converted.columns 
                        if col.startswith('f_') and df_converted[col].dtype.name == 'category']
labels = [col for col in df_converted.columns if col.startswith('l_')]

# 1. Sélection des features numériques (inchangé)
corr_sums = {feature: sum(abs(df_converted[feature].corr(df_converted[label])) for label in labels)
             for feature in numeric_features}
sorted_features = sorted(corr_sums.items(), key=lambda x: x[1], reverse=True)
selected_numeric = []
for feature, _ in sorted_features:
    if not selected_numeric:
        selected_numeric.append(feature)
    else:
        if all(abs(df_converted[feature].corr(df_converted[f])) <= 0.85 for f in selected_numeric):
            selected_numeric.append(feature)
print("Features numériques sélectionnées:", selected_numeric)

# 2. Sélection des features catégorielles
#    calcul de score (Wilcoxon ou Kruskal) inchangé
categorical_scores = {}
for feature in categorical_features:
    total_score = 0
    for label in labels:
        groups = [g[label].values for _, g in df_converted.groupby(feature, observed=False)]
        if len(groups) == 2:
            _, p = stats.mannwhitneyu(groups[0], groups[1], alternative='two-sided')
        else:
            _, p = stats.kruskal(*groups)
        total_score += -np.log(p) if p > 0 else 1000
    categorical_scores[feature] = total_score

sorted_categorical = [f for f, _ in sorted(categorical_scores.items(), key=lambda x: x[1], reverse=True)]
to_eliminate = set()

# Boucle d'élimination basée sur dépendance entre paires
for i in range(len(sorted_categorical)):
    f1 = sorted_categorical[i]
    if f1 in to_eliminate:
        continue
    for f2 in sorted_categorical[i+1:]:
        if f2 in to_eliminate:
            continue

        # 2×2 ? Fisher : sinon Chi²
        contingency = pd.crosstab(df_converted[f1], df_converted[f2])
        if contingency.shape == (2, 2):
            _, p_val = fisher_exact(contingency)
        else:
            _, p_val, _, _ = chi2_contingency(contingency)

        if p_val < 0.05:
            # on élimine la moins « informative »
            if categorical_scores[f1] >= categorical_scores[f2]:
                to_eliminate.add(f2)
            else:
                to_eliminate.add(f1)
                break

selected_categorical = [f for f in sorted_categorical if f not in to_eliminate]
print("Features catégorielles sélectionnées:", selected_categorical)

# 3. Dernière passe : dépendance catégorielle vs numérique (inchangé)
categorical_to_eliminate = set()
for cat in selected_categorical:
    for num in selected_numeric:
        groups = [g[num].values for _, g in df_converted.groupby(cat, observed=False)]
        if len(groups) == 2:
            _, p = stats.mannwhitneyu(groups[0], groups[1], alternative='two-sided')
        else:
            _, p = stats.kruskal(*groups)
        if p < 0.05:
            categorical_to_eliminate.add(cat)
            break

final_categorical = [f for f in selected_categorical if f not in categorical_to_eliminate]

# Mise à jour des ensembles globaux
features_to_keep.update(selected_numeric + final_categorical)
features_to_eliminate.update(
    [f for f in numeric_features if f not in selected_numeric] +
    [f for f in categorical_features if f not in final_categorical]
)

# Résumé
print("\nRésultats finaux:")
print(" → Conservées :", selected_numeric + final_categorical)
print(" → Éliminées :", sorted(to_eliminate | categorical_to_eliminate))

# Affichage d’un tableau récapitulatif
summary = []
for feat in numeric_features + categorical_features:
    status = "Conservée" if feat in features_to_keep else "Éliminée"
    if feat in selected_numeric:
        reason = "Corrélation élevée avec labels"
    elif feat in final_categorical:
        reason = "Score élevé et indépendance validée"
    elif feat in to_eliminate:
        reason = "Dépendance entre catégorielles"
    elif feat in categorical_to_eliminate:
        reason = "Dépendance avec numérique"
    else:
        reason = "Score insuffisant"
    summary.append({
        'Feature': feat,
        'Type': 'Numérique' if feat in numeric_features else 'Catégorielle',
        'Status': status,
        'Raison': reason
    })
summary_df = pd.DataFrame(summary)
display(
    summary_df.style
        .set_properties(**{'text-align': 'left'})
        .set_table_styles([
            {'selector': 'th', 'props': [('text-align', 'left')]},
            {'selector': 'td', 'props': [('text-align', 'left')]}
        ])
        .hide(axis='index')
)

# Enfin, construire le dictionnaire filtré
cols_to_drop = [f for f in df_converted.columns if f in features_to_eliminate]
df_selected = df_converted.drop(columns=cols_to_drop)



Sélection des features pour normalized_ENB2012_data.csv
Features numériques sélectionnées: ['f_overall_height', 'f_relative_compactness', 'f_wall_area', 'f_glazing_area']
Features catégorielles sélectionnées: ['f_glazing_distribution', 'f_orientation']

Résultats finaux:
 → Conservées : ['f_overall_height', 'f_relative_compactness', 'f_wall_area', 'f_glazing_area', 'f_orientation']
 → Éliminées : ['f_glazing_distribution']


Feature,Type,Status,Raison
f_relative_compactness,Numérique,Conservée,Corrélation élevée avec labels
f_surface_area,Numérique,Éliminée,Score insuffisant
f_wall_area,Numérique,Conservée,Corrélation élevée avec labels
f_roof_area,Numérique,Éliminée,Score insuffisant
f_overall_height,Numérique,Conservée,Corrélation élevée avec labels
f_glazing_area,Numérique,Conservée,Corrélation élevée avec labels
f_orientation,Catégorielle,Conservée,Score élevé et indépendance validée
f_glazing_distribution,Catégorielle,Éliminée,Dépendance avec numérique


## 4. Définitions des fonctions de transformation

In [58]:
# vos autres transform et inverse-transform
TRANSFORM_FNS = {
    'none':   lambda x: x,
    'log':    np.log,
    'log1p':  np.log1p,
    'sqrt':   np.sqrt,
    'cbrt':   np.cbrt,
    'square': lambda x: x**2,
    'cube':   lambda x: x**3,
    'power4': lambda x: x**4,
}

INV_TRANSFORM_FNS = {
    'none':   lambda x: x,
    'log':    np.exp,
    'log1p':  np.expm1,
    'sqrt':   lambda x: x**2,
    'cbrt':   lambda x: x**3,
    'square': np.sqrt,
    'cube':   np.cbrt,
    'power4': lambda x: x**(1/4),
}

def apply_transform(series: pd.Series, method: str):
    """
    Retourne soit:
      - (serie_t, lambda) pour boxcox
      - serie_t pour les autres méthodes
    """
    if method == 'boxcox':
        # on s'assure que la série est strictement positive
        if (series <= 0).any():
            shift = -series.min() + 1e-6
            series = series + shift
            print(f"⚠ {method}: data shifted by +{shift:.6f} pour être >0")
        transformed, lmbda = boxcox(series.values)
        return transformed, lmbda
    else:
        fn = TRANSFORM_FNS.get(method)
        if fn is None:
            raise ValueError(f"Transformation inconnue: {method}")
        return fn(series.values)

def inverse_transform(y, method: str, lmbda=None):
    """
    Applique l'inverse de la transformation.
    Pour boxcox, il faut fournir λ.
    """
    if method == 'boxcox':
        if lmbda is None:
            raise ValueError("λ manquant pour l'inverse de boxcox")
        # inverse analytique
        y = np.asarray(y)
        if lmbda == 0:
            return np.exp(y)
        else:
            return (y * lmbda + 1) ** (1.0 / lmbda)
    else:
        fn = INV_TRANSFORM_FNS.get(method)
        if fn is None:
            raise ValueError(f"Transformation inverse inconnue: {method}")
        return fn(y)

l_transformations = ['log', 'sqrt', 'square', 'cube', 'power4', 'cbrt']
#f_transformations = ['log', 'sqrt', 'square']
#f_transformations = ['boxcox', 'log', 'sqrt', 'square', 'cube', 'power4', 'cbrt']#KO
#f_transformations = ['boxcox', 'log', 'sqrt', 'square']#KO
f_transformations = ['log', 'sqrt', 'square']#OK
#f_transformations = ['log', 'sqrt', 'square', 'cube', 'power4', 'cbrt']#KO
#f_transformations = ['log', 'sqrt', 'square', 'cube', 'cbrt']#KO
#f_transformations = ['log', 'sqrt', 'square', 'cube']#KO
#f_transformations = ['log', 'sqrt', 'square', 'cbrt']#KO
#f_transformations = ['log', 'sqrt', 'square', 'power4']#KO
#f_transformations = ['log', 'sqrt', 'power4']#KO
#f_transformations = ['log', 'sqrt', 'cube']#KO
#f_transformations = ['sqrt','square', 'cube']#KO
#f_transformations = ['log','square', 'cube']#KO
#l_transformations = ['log', 'sqrt', 'square']

dict_l_transfo = {"l_cooling_load":"log",
                  "l_heating_load":"log"}
label_transformer = FunctionTransformer(
    func=apply_transform,
    inverse_func=inverse_transform,
    validate=False
)


## 5. Transformation des labels

### Objectif
Transformer les labels.


In [59]:
print("=" * 80)
print(f"Transformation des labels")
print("=" * 80)


# Copie du DataFrame pour stocker les labels transformés
df_transformed = df_selected.copy()

for label,method in dict_l_transfo.items():
    if method != "none":
        print(f"Transformation de {label} avec {method}")
        try:
            if method == "boxcox":
                # apply_transform renvoie (série_t, λ)
                series_t, lmbda = apply_transform(df_selected[label], "boxcox")
                boxcox_lambdas[label] = lmbda
            else:
                series_t = apply_transform(df_selected[label], method)
    
            # On remplace la colonne transformée
            df_transformed[label+"_orig"] = df_transformed[label]
            df_transformed[label] = series_t
        except Exception as e:
            print(f"⚠ Erreur pendant la transformation {method} de {label} : {e}")

df_transformed

Transformation des labels
Transformation de l_cooling_load avec log
Transformation de l_heating_load avec log


,f_relative_compactness,f_wall_area,f_overall_height,f_orientation,f_glazing_area,l_heating_load,l_cooling_load,l_cooling_load_orig,l_heating_load_orig
0,0.98,294.0,7.0,2,0.0,2.744061,3.060115,21.33,15.55
1,0.98,294.0,7.0,3,0.0,2.744061,3.060115,21.33,15.55
2,0.98,294.0,7.0,4,0.0,2.744061,3.060115,21.33,15.55
3,0.98,294.0,7.0,5,0.0,2.744061,3.060115,21.33,15.55
4,0.90,318.5,7.0,2,0.0,3.036874,3.342155,28.28,20.84
...,...,...,...,...,...,...,...,...,...
763,0.64,343.0,3.5,5,0.4,2.883683,3.063391,21.40,17.88
764,0.62,367.5,3.5,2,0.4,2.805782,2.826129,16.88,16.54
765,0.62,367.5,3.5,3,0.4,2.799717,2.839663,17.11,16.44
766,0.62,367.5,3.5,4,0.4,2.802148,2.810005,16.61,16.48


## 6. Fonction entraînement

In [60]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

In [63]:
import inspect
from pathlib import Path

# Support XGBoost silence
try:
    import xgboost as xgb
    xgb.set_config(verbosity=0)
except ImportError:
    xgb = None

# Support LightGBM
try:
    from lightgbm import LGBMRegressor
except ImportError:
    LGBMRegressor = None

warnings.filterwarnings("ignore", category=UserWarning)


def _manual_cv_lgbm(X, y, params, cv, random_state, early_stopping_rounds):
    """
    Réalise une CV manuelle avec early stopping pour un jeu de params LightGBM.
    Retourne la RMSE moyenne.
    """
    rmses = []
    kf = KFold(n_splits=cv, shuffle=True, random_state=random_state)

    for train_idx, val_idx in kf.split(X):
        X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
        # Slice compatible Series ou ndarray
        y_tr = y[train_idx] if isinstance(y, np.ndarray) else y.iloc[train_idx]
        y_val = y[val_idx]   if isinstance(y, np.ndarray) else y.iloc[val_idx]

        m = LGBMRegressor(
            random_state=random_state,
            n_jobs=-1,
            verbose=-1,
            verbosity=-1,
            force_col_wise=True,
            **params
        )
        m.fit(
            X_tr, y_tr,
            eval_set=[(X_val, y_val)],
            callbacks=[lgb.early_stopping(stopping_rounds=early_stopping_rounds),lgb.log_evaluation(0)],
            
        )        
        y_pred = m.predict(X_val)
        rmses.append(np.sqrt(mean_squared_error(y_val, y_pred)))

    return np.mean(rmses)


def train_models(
    df,
    model_cls,
    param_grid,
    apply_transform,
    inverse_transform,
    boxcox_lambdas,
    test_size=0.2,
    cv=5,
    random_state=42,
    apply_lgbm_adjustments=True
):
    results = {}
    save_dir = Path('saved_models')
    save_dir.mkdir(exist_ok=True)

    orig_labels = [c for c in df.columns if c.startswith('l_') and c.endswith('_orig')]

    for label_orig in orig_labels:
        label = label_orig[:-5]

        # 1) Transformation y
        trafo = dict_l_transfo[label]
        y_orig = df[label_orig]
        if trafo == 'boxcox':
            y_trf, lmbda = apply_transform(y_orig, 'boxcox')
            boxcox_lambdas[label] = lmbda
        else:
            y_trf = apply_transform(y_orig, trafo)
            lmbda = None

        # 2) Préparation X
        feat_cols = [c for c in df.columns if c.startswith('f_')]
        X = df[feat_cols]
        numeric_feats = [c for c in feat_cols if pd.api.types.is_numeric_dtype(df[c])]
        categorical_feats = [c for c in feat_cols if not pd.api.types.is_numeric_dtype(df[c])]
        preproc = ColumnTransformer([
            ('num', StandardScaler(), numeric_feats),
            ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_feats)
        ])

        # 3) Split
        X_tr, X_te, y_tr, y_te = train_test_split(
            X, y_trf,
            test_size=test_size,
            random_state=random_state
        )
        # 4) Initialisation dynamique
        init_params = inspect.signature(model_cls.__init__).parameters
        model_kwargs = {}
        
        # random_state & n_jobs pour tous
        if 'random_state' in init_params:
            model_kwargs['random_state'] = random_state
        if 'n_jobs' in init_params:
            model_kwargs['n_jobs'] = -1
        
        # verbose/verbosity pour LGBM seulement
        if LGBMRegressor and issubclass(model_cls, LGBMRegressor):
            if 'verbose' in init_params:
                model_kwargs['verbose'] = -1
            if 'verbosity' in init_params:
                model_kwargs['verbosity'] = -1
        if xgb and hasattr(xgb.XGBRegressor, '__init__') and issubclass(model_cls, xgb.XGBRegressor):
            # XGBRegressor accepte 'verbosity', pas 'verbose'
            if 'verbosity' in init_params:
                model_kwargs['verbosity'] = 0

        # 5) Cas LightGBM spécifique. On ne va pas utiliser la pipeline classique ni la gridsearch classique 
        #car ça empêche de faire de l'early stopping et rend l'entraînement extrêmement long.
        #On va donc coder chaque étape.
        if (apply_lgbm_adjustments
                and LGBMRegressor
                and issubclass(model_cls, LGBMRegressor)):

            # a) Préparer la grille et transformer X. On supprime le max depth = -1 qui peut pénaliser et fausser le LightGBM
            
            local_grid = param_grid.copy()
            if 'model__max_depth' in local_grid:
                vals = [d for d in local_grid['model__max_depth'] if d != -1]
                local_grid['model__max_depth'] = vals or local_grid['model__max_depth']
            plain_grid = {k.replace('model__', ''): v for k, v in local_grid.items()}

            preproc.fit(X_tr)
            X_tr_t = preproc.transform(X_tr)
            X_te_t = preproc.transform(X_te)

            # Récupérer noms de colonnes
            cols_num = preproc.named_transformers_['num'].get_feature_names_out(numeric_feats)
            cols_cat = preproc.named_transformers_['cat'].get_feature_names_out(categorical_feats)
            feat_names = np.concatenate([cols_num, cols_cat])
            X_tr_t = pd.DataFrame(X_tr_t, columns=feat_names, index=X_tr.index)
            X_te_t = pd.DataFrame(X_te_t, columns=feat_names, index=X_te.index)

            # b) CV manuelle
            best_mean = np.inf
            best_params = None
            for params in ParameterGrid(plain_grid):
                mean_rmse = _manual_cv_lgbm(
                    X_tr_t, y_tr,
                    params,
                    cv=cv,
                    random_state=random_state,
                    early_stopping_rounds=10
                )
                if mean_rmse < best_mean:
                    best_mean = mean_rmse
                    best_params = params

            cv_rmse_trf = best_mean

            # c) Entraînement final
            final_model = model_cls(**{**model_kwargs, **best_params})
            final_model.fit(
                X_tr_t, y_tr,
                eval_set=[(X_te_t, y_te)],
                callbacks=[lgb.early_stopping(stopping_rounds=10),lgb.log_evaluation(0)]
            )
            best_pipe = Pipeline([('preproc', preproc), ('model', final_model)])

        else:
            # 6) Pipeline standard + GridSearchCV
            pipe = Pipeline([
                ('preproc', preproc),
                ('model', model_cls(**model_kwargs))
            ])
            gs = GridSearchCV(
                estimator=pipe,
                param_grid=param_grid,
                cv=cv,
                scoring='neg_root_mean_squared_error',
                n_jobs=-1,
                verbose=0
            )
            gs.fit(X_tr, y_tr)
            best_pipe = gs.best_estimator_
            best_params = gs.best_params_
            cv_rmse_trf = -gs.best_score_

        # 7) Évaluation sur le test set
        y_pred_trf = best_pipe.predict(X_te)
        if trafo == 'boxcox' and lmbda is not None:
            y_te_orig = inverse_transform(y_te, 'boxcox', lmbda)
            y_pred_orig = inverse_transform(y_pred_trf, 'boxcox', lmbda)
        elif trafo != 'none':
            y_te_orig = inverse_transform(y_te, trafo)
            y_pred_orig = inverse_transform(y_pred_trf, trafo)
        else:
            y_te_orig, y_pred_orig = y_te, y_pred_trf

        test_rmse = np.sqrt(mean_squared_error(y_te_orig, y_pred_orig))

        # 8) Sauvegarde
        fname_base = f"{model_cls.__name__.lower()}_{file_name}_{label}"
        try:
            import joblib
            joblib.dump(best_pipe, save_dir / f"{fname_base}.joblib")
            print(f"[{file_name}][{label}] Pipeline saved: {fname_base}.joblib")
        except Exception as e:
            print(f"[{file_name}][{label}] Joblib save error: {e}")

        model = best_pipe.named_steps['model']
        try:
            if hasattr(model, 'booster_'):
                model.booster_.save_model(str(save_dir / f"{fname_base}_booster.txt"))
                print(f"[{file_name}][{label}] Booster saved: {fname_base}_booster.txt")
            elif hasattr(model, 'get_booster'):
                model.get_booster().save_model(str(save_dir / f"{fname_base}_booster.txt"))
                print(f"[{file_name}][{label}] Booster saved: {fname_base}_booster.txt")
        except Exception as e:
            print(f"[{file_name}][{label}] Booster save error: {e}")

        # 9) Stockage résultats et prints finaux
        results[(file_name, label)] = {
            'model': best_pipe,
            'best_params': best_params,
            'cv_rmse_trf': cv_rmse_trf,
            'test_rmse_orig': test_rmse,
            'transformation': trafo,
            'lambda': lmbda
        }
        print(f"[{file_name}][{label}] Best params: {best_params}")
        print(f"CV RMSE(trf)={cv_rmse_trf:.4f}, Test RMSE(orig)={test_rmse:.4f}")

    return results


## 7. Entrainement XGBoost

In [64]:
# Exemple d'utilisation:
param_grid_xgb = {
#    'model__n_estimators':      [100, 200, 300],
    'model__n_estimators':      [50, 100, 300],
#    'model__max_depth':         [3, 6, 9],
    'model__max_depth':         [2, 3, 4],
#    'model__learning_rate':     [0.01, 0.1, 0.2],
    'model__learning_rate':     [0.1, 0.15, 0.2],
#    'model__subsample':         [0.6, 0.8, 1.0],
    'model__subsample':         [0.9,0.95, 1.0],
#    'model__colsample_bytree':  [0.6, 0.8, 1.0]
    'model__colsample_bytree':  [0.5,0.6, 0.7]
}
xgb_results = train_models(df_transformed,
                                XGBRegressor,
                                param_grid_xgb,
                                apply_transform,
                                inverse_transform,
                                boxcox_lambdas)


[normalized_ENB2012_data.csv][l_cooling_load] Pipeline saved: xgbregressor_normalized_ENB2012_data.csv_l_cooling_load.joblib
[normalized_ENB2012_data.csv][l_cooling_load] Booster saved: xgbregressor_normalized_ENB2012_data.csv_l_cooling_load_booster.txt
[normalized_ENB2012_data.csv][l_cooling_load] Best params: {'model__colsample_bytree': 0.7, 'model__learning_rate': 0.2, 'model__max_depth': 2, 'model__n_estimators': 300, 'model__subsample': 1.0}
CV RMSE(trf)=0.0504, Test RMSE(orig)=1.7471
[normalized_ENB2012_data.csv][l_heating_load] Pipeline saved: xgbregressor_normalized_ENB2012_data.csv_l_heating_load.joblib
[normalized_ENB2012_data.csv][l_heating_load] Booster saved: xgbregressor_normalized_ENB2012_data.csv_l_heating_load_booster.txt
[normalized_ENB2012_data.csv][l_heating_load] Best params: {'model__colsample_bytree': 0.7, 'model__learning_rate': 0.15, 'model__max_depth': 3, 'model__n_estimators': 300, 'model__subsample': 1.0}
CV RMSE(trf)=0.0241, Test RMSE(orig)=0.5267


## 8. Entrainement LightGBM

In [65]:

from lightgbm import LGBMRegressor
param_grid_lgbm = {
#    'model__n_estimators':      [100, 200, 300],
    'model__n_estimators':      [50, 100, 300],
#    'model__max_depth':         [-1, 5, 10],
#    'model__max_depth':         [2, 5, 8],
    'model__max_depth':         [2, 5],
#    'model__learning_rate':     [0.01, 0.1, 0.2],
    'model__learning_rate':     [0.1, 0.15, 0.2],
#    'model__num_leaves':        [31, 63, 127],
#    'model__num_leaves':        [15, 31, 47],
    'model__num_leaves':        [12, 15, 22],
#    'model__subsample':         [0.6, 0.8, 1.0]
    'model__subsample':         [0.4, 0.6, 0.7]
}
lgbm_results = train_models(
    df_transformed,
    LGBMRegressor,
    param_grid_lgbm,
    apply_transform,
    inverse_transform,
    boxcox_lambdas
)

Training until validation scores don't improve for 10 rounds
Did not meet early stopping. Best iteration is:
[50]	valid_0's l2: 0.00486366
Training until validation scores don't improve for 10 rounds
Did not meet early stopping. Best iteration is:
[50]	valid_0's l2: 0.00382671
Training until validation scores don't improve for 10 rounds
Did not meet early stopping. Best iteration is:
[50]	valid_0's l2: 0.00541757
Training until validation scores don't improve for 10 rounds
Did not meet early stopping. Best iteration is:
[50]	valid_0's l2: 0.00470475
Training until validation scores don't improve for 10 rounds
Did not meet early stopping. Best iteration is:
[50]	valid_0's l2: 0.00460018
Training until validation scores don't improve for 10 rounds
Did not meet early stopping. Best iteration is:
[50]	valid_0's l2: 0.00486366
Training until validation scores don't improve for 10 rounds
Did not meet early stopping. Best iteration is:
[50]	valid_0's l2: 0.00382671
Training until validation s

In [67]:
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.gaussian_process import GaussianProcessRegressor

In [68]:
# 1) Random Forest
param_grid_rf = {
    'model__n_estimators': [100, 300, 500],
    'model__max_depth':    [None, 10, 20],
    'model__max_features': ['sqrt', 'log2']
}
rf_results = train_models(
    df_transformed,
    RandomForestRegressor,
    param_grid_rf,
    apply_transform,
    inverse_transform,
    boxcox_lambdas
)

[normalized_ENB2012_data.csv][l_cooling_load] Pipeline saved: randomforestregressor_normalized_ENB2012_data.csv_l_cooling_load.joblib
[normalized_ENB2012_data.csv][l_cooling_load] Best params: {'model__max_depth': 20, 'model__max_features': 'log2', 'model__n_estimators': 500}
CV RMSE(trf)=0.0611, Test RMSE(orig)=1.9780
[normalized_ENB2012_data.csv][l_heating_load] Pipeline saved: randomforestregressor_normalized_ENB2012_data.csv_l_heating_load.joblib
[normalized_ENB2012_data.csv][l_heating_load] Best params: {'model__max_depth': None, 'model__max_features': 'log2', 'model__n_estimators': 500}
CV RMSE(trf)=0.0448, Test RMSE(orig)=0.7551


In [69]:
# 2) Extra Trees
param_grid_et = {
    'model__n_estimators': [100, 300, 500],
    'model__max_depth':    [None, 10, 20],
    'model__max_features': ['sqrt', 'log2']
}
et_results = train_models(
    df_transformed,
    ExtraTreesRegressor,
    param_grid_et,
    apply_transform,
    inverse_transform,
    boxcox_lambdas
)


[normalized_ENB2012_data.csv][l_cooling_load] Pipeline saved: extratreesregressor_normalized_ENB2012_data.csv_l_cooling_load.joblib
[normalized_ENB2012_data.csv][l_cooling_load] Best params: {'model__max_depth': 10, 'model__max_features': 'log2', 'model__n_estimators': 100}
CV RMSE(trf)=0.0615, Test RMSE(orig)=1.9848
[normalized_ENB2012_data.csv][l_heating_load] Pipeline saved: extratreesregressor_normalized_ENB2012_data.csv_l_heating_load.joblib
[normalized_ENB2012_data.csv][l_heating_load] Best params: {'model__max_depth': 10, 'model__max_features': 'log2', 'model__n_estimators': 300}
CV RMSE(trf)=0.0493, Test RMSE(orig)=0.8450


In [72]:
# 3) Ridge Regression
param_grid_ridge = {
    'model__alpha': [0.01, 0.1, 1, 10]
}
ridge_results = train_models(
    df_transformed,
    Ridge,
    param_grid_ridge,
    apply_transform,
    inverse_transform,
    boxcox_lambdas
)

[normalized_ENB2012_data.csv][l_cooling_load] Pipeline saved: ridge_normalized_ENB2012_data.csv_l_cooling_load.joblib
[normalized_ENB2012_data.csv][l_cooling_load] Best params: {'model__alpha': 1}
CV RMSE(trf)=0.1131, Test RMSE(orig)=3.2075
[normalized_ENB2012_data.csv][l_heating_load] Pipeline saved: ridge_normalized_ENB2012_data.csv_l_heating_load.joblib
[normalized_ENB2012_data.csv][l_heating_load] Best params: {'model__alpha': 1}
CV RMSE(trf)=0.1221, Test RMSE(orig)=3.1696


In [71]:
# 4) Lasso Regression
param_grid_lasso = {
    'model__alpha': [0.01, 0.1, 1, 10]
}
lasso_results = train_models(
    df_transformed,
    Lasso,
    param_grid_lasso,
    apply_transform,
    inverse_transform,
    boxcox_lambdas
)

[normalized_ENB2012_data.csv][l_cooling_load] Pipeline saved: lasso_normalized_ENB2012_data.csv_l_cooling_load.joblib
[normalized_ENB2012_data.csv][l_cooling_load] Best params: {'model__alpha': 0.01}
CV RMSE(trf)=0.1149, Test RMSE(orig)=3.3523
[normalized_ENB2012_data.csv][l_heating_load] Pipeline saved: lasso_normalized_ENB2012_data.csv_l_heating_load.joblib
[normalized_ENB2012_data.csv][l_heating_load] Best params: {'model__alpha': 0.01}
CV RMSE(trf)=0.1225, Test RMSE(orig)=3.1644


In [73]:
# 5) ElasticNet
param_grid_enet = {
    'model__alpha': [0.01, 0.1, 1, 10],
    'model__l1_ratio': [0.0, 0.5, 1.0]
}
enet_results = train_models(
    df_transformed,
    ElasticNet,
    param_grid_enet,
    apply_transform,
    inverse_transform,
    boxcox_lambdas
)

[normalized_ENB2012_data.csv][l_cooling_load] Pipeline saved: elasticnet_normalized_ENB2012_data.csv_l_cooling_load.joblib
[normalized_ENB2012_data.csv][l_cooling_load] Best params: {'model__alpha': 0.01, 'model__l1_ratio': 0.0}
CV RMSE(trf)=0.1135, Test RMSE(orig)=3.2654
[normalized_ENB2012_data.csv][l_heating_load] Pipeline saved: elasticnet_normalized_ENB2012_data.csv_l_heating_load.joblib
[normalized_ENB2012_data.csv][l_heating_load] Best params: {'model__alpha': 0.01, 'model__l1_ratio': 0.5}
CV RMSE(trf)=0.1219, Test RMSE(orig)=3.1706


/opt/conda/lib/python3.11/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 3.376e+00, tolerance: 7.595e-03 Linear regression models with null weight for the l1 regularization term are more efficiently fitted using one of the solvers implemented in sklearn.linear_model.Ridge/RidgeCV instead.
  model = cd_fast.enet_coordinate_descent(
/opt/conda/lib/python3.11/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 3.534e+00, tolerance: 7.607e-03 Linear regression models with null weight for the l1 regularization term are more efficiently fitted using one of the solvers implemented in sklearn.linear_mode

In [74]:
# 6) Support Vector Regression
param_grid_svr = {
    'model__C': [0.1, 1, 10],
    'model__gamma': ['scale', 'auto']
}
svr_results = train_models(
    df_transformed,
    SVR,
    param_grid_svr,
    apply_transform,
    inverse_transform,
    boxcox_lambdas
)

[normalized_ENB2012_data.csv][l_cooling_load] Pipeline saved: svr_normalized_ENB2012_data.csv_l_cooling_load.joblib
[normalized_ENB2012_data.csv][l_cooling_load] Best params: {'model__C': 10, 'model__gamma': 'scale'}
CV RMSE(trf)=0.0816, Test RMSE(orig)=2.3032
[normalized_ENB2012_data.csv][l_heating_load] Pipeline saved: svr_normalized_ENB2012_data.csv_l_heating_load.joblib
[normalized_ENB2012_data.csv][l_heating_load] Best params: {'model__C': 10, 'model__gamma': 'scale'}
CV RMSE(trf)=0.0789, Test RMSE(orig)=1.9109


In [75]:
# 7) K-Nearest Neighbors
param_grid_knn = {
    'model__n_neighbors': [3, 5, 10,20,30],
    'model__weights': ['uniform', 'distance']
}
knn_results = train_models(
    df_transformed,
    KNeighborsRegressor,
    param_grid_knn,
    apply_transform,
    inverse_transform,
    boxcox_lambdas
)

[normalized_ENB2012_data.csv][l_cooling_load] Pipeline saved: kneighborsregressor_normalized_ENB2012_data.csv_l_cooling_load.joblib
[normalized_ENB2012_data.csv][l_cooling_load] Best params: {'model__n_neighbors': 3, 'model__weights': 'distance'}
CV RMSE(trf)=0.0717, Test RMSE(orig)=2.0928
[normalized_ENB2012_data.csv][l_heating_load] Pipeline saved: kneighborsregressor_normalized_ENB2012_data.csv_l_heating_load.joblib
[normalized_ENB2012_data.csv][l_heating_load] Best params: {'model__n_neighbors': 3, 'model__weights': 'distance'}
CV RMSE(trf)=0.0876, Test RMSE(orig)=1.2918


In [76]:
# 8) Multi-layer Perceptron
param_grid_mlp = {
    'model__hidden_layer_sizes': [(50,), (100, 50), (100, 100, 50)],
    'model__alpha': [1e-4, 1e-3],
    'model__learning_rate_init': [1e-3, 1e-4]
}
mlp_results = train_models(
    df_transformed,
    MLPRegressor,
    param_grid_mlp,
    apply_transform,
    inverse_transform,
    boxcox_lambdas
)

/opt/conda/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/opt/conda/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/opt/conda/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/opt/conda/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/opt/conda/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptro

[normalized_ENB2012_data.csv][l_cooling_load] Pipeline saved: mlpregressor_normalized_ENB2012_data.csv_l_cooling_load.joblib
[normalized_ENB2012_data.csv][l_cooling_load] Best params: {'model__alpha': 0.001, 'model__hidden_layer_sizes': (100, 100, 50), 'model__learning_rate_init': 0.001}
CV RMSE(trf)=0.1089, Test RMSE(orig)=2.5409


/opt/conda/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/opt/conda/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/opt/conda/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/opt/conda/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/opt/conda/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptro

[normalized_ENB2012_data.csv][l_heating_load] Pipeline saved: mlpregressor_normalized_ENB2012_data.csv_l_heating_load.joblib
[normalized_ENB2012_data.csv][l_heating_load] Best params: {'model__alpha': 0.001, 'model__hidden_layer_sizes': (100, 100, 50), 'model__learning_rate_init': 0.001}
CV RMSE(trf)=0.1004, Test RMSE(orig)=2.0583


In [77]:
# 9) Gaussian Process Regression
param_grid_gpr = {
    'model__alpha': [1e-10, 1e-5, 1e-2]
}
gpr_results = train_models(
    df_transformed,
    GaussianProcessRegressor,
    param_grid_gpr,
    apply_transform,
    inverse_transform,
    boxcox_lambdas
)

[normalized_ENB2012_data.csv][l_cooling_load] Pipeline saved: gaussianprocessregressor_normalized_ENB2012_data.csv_l_cooling_load.joblib
[normalized_ENB2012_data.csv][l_cooling_load] Best params: {'model__alpha': 1e-10}
CV RMSE(trf)=0.0670, Test RMSE(orig)=2.0052
[normalized_ENB2012_data.csv][l_heating_load] Pipeline saved: gaussianprocessregressor_normalized_ENB2012_data.csv_l_heating_load.joblib
[normalized_ENB2012_data.csv][l_heating_load] Best params: {'model__alpha': 1e-10}
CV RMSE(trf)=0.0416, Test RMSE(orig)=0.6936
